## Category important words & similarity search

In [59]:
import pandas as pd
import numpy as np
import re
import time
import nltk
#from nltk import bigrams, trigrams
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity


#model = SentenceTransformer('/Users/zphilipp/git/research/relevance/models/sentence-transformer.model')
model = SentenceTransformer('all-MiniLM-L6-v2')

#pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 200)

prepositions_and_conjunctions = [
    "about", "above", "across", "after", "against", "along", "among", "around", "at",
    "before", "behind", "below", "beneath", "beside", "between", "beyond", "by",
    "during", "for", "from", "in", "inside", "into", "near", "of", "off", "on",
    "out", "outside", "over", "through", "throughout", "to", "toward", "under",
    "until", "up", "with", "within", "without", "and", "but", "or", "for", "nor",
    "so", "yet", "although", "because", "as", "since", "unless", "while", "when",
    "where", "after", "before", "the", "a"
]
pattern = r'\b(?:' + '|'.join(prepositions_and_conjunctions) + r')\b'

def remove_prepositions_and_conjunctions(text):
    text = text.lower()
    cleaned_text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r'\d+', '', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    return cleaned_text.replace("-", "")

/Users/zphilipp/miniconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


#### Get all titles from Deals and Options text

In [60]:
df = pd.read_csv('category_data.csv').dropna()#.head()
df['text'] = df['description'] + ' ' + df['name']# + ' ' + df['category']
df['text'] = df['text'].apply(remove_prepositions_and_conjunctions)
df_ = df
df_.head()

,attributes.v2_category_name,attributes.v1_category_name,parent,guid,description,name,header,category,text
0,Shopping,Retail,0ed8f46e-2990-448c-9a8c-50665498a84c,7552494f-2a02-4bf8-91b3-ba34d90debdf,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Personal Care,Shopping,health & beauty deodorant & antiperspirant mens deodorant health & beauty deodorant & antiperspirant mens deodorant
2,Shopping,Retail,71912b5b-4809-4536-bc03-8ee270060b96,36a0ea5b-210a-452f-82ad-e9b6c96c67ab,Home & Garden - Decorative Vases,Home & Garden - Decorative Vases,Home Decor,Shopping,home & garden decorative vases home & garden decorative vases
3,Shopping,Retail,8efb9d2c-0947-44d9-af8f-eec79a5a3fc2,2073934d-da10-4cdf-bed1-59bb12e9d00a,Apparel & Accessories - Girls - Belts,Apparel & Accessories - Girls - Belts,Clothing Accessories,Shopping,apparel & accessories girls belts apparel & accessories girls belts
4,Shopping,Retail,d05bb230-5f44-4180-9f9b-323f72649552,508d94c4-3851-4b51-b0c0-4efcf6381fcb,Action Figure / Doll,Action Figure / Doll,Toys & Hobbies,Shopping,action figure / doll action figure / doll
5,Travel,Channel - Travel,5d99d471-fb8a-496a-843b-f0a4a75a6e30,a63aa87d-7f06-47ca-bbc7-8197f773fc1e,Tour - Hokkaido & Tohoku - Thermal Bath,Tour - Hokkaido & Tohoku - Thermal Bath,Asia,Travel,tour hokkaido & tohoku thermal bath tour hokkaido & tohoku thermal bath


#### Create word embedings and transform data

In [61]:
df_['text'] = df_['text'].tolist()
df_['text_embeddings'] = df_['text'].apply(lambda x: model.encode(x))

combined_embeddings = np.array(df_['text_embeddings'].tolist())

In [62]:
df_.head()

,attributes.v2_category_name,attributes.v1_category_name,parent,guid,description,name,header,category,text,text_embeddings
0,Shopping,Retail,0ed8f46e-2990-448c-9a8c-50665498a84c,7552494f-2a02-4bf8-91b3-ba34d90debdf,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Personal Care,Shopping,health & beauty deodorant & antiperspirant mens deodorant health & beauty deodorant & antiperspirant mens deodorant,"[-0.009740936, 0.01637392, 0.10180105, 0.015788225, 0.08613156, -0.05136446, 0.10834356, -0.023668014, -0.055397417, -0.020835223, 0.07027916, -0.053843062, -0.018749349, -0.05474668, 0.09677764, ..."
2,Shopping,Retail,71912b5b-4809-4536-bc03-8ee270060b96,36a0ea5b-210a-452f-82ad-e9b6c96c67ab,Home & Garden - Decorative Vases,Home & Garden - Decorative Vases,Home Decor,Shopping,home & garden decorative vases home & garden decorative vases,"[0.050594825, 0.021106621, 0.0492115, -0.08283868, -0.037469223, 0.042464323, 0.053372044, -0.012485839, -0.04804874, 0.034488536, -0.055529684, -0.036089577, -0.01227808, 0.043490466, 0.086594746..."
3,Shopping,Retail,8efb9d2c-0947-44d9-af8f-eec79a5a3fc2,2073934d-da10-4cdf-bed1-59bb12e9d00a,Apparel & Accessories - Girls - Belts,Apparel & Accessories - Girls - Belts,Clothing Accessories,Shopping,apparel & accessories girls belts apparel & accessories girls belts,"[0.008360008, 0.00015190663, 0.008499597, 0.011574635, -0.024196286, -0.056413222, 0.096690245, -0.017104944, -0.048308298, 0.004226838, 0.1234592, -0.018754913, 0.11234592, -0.042638365, 0.053102..."
4,Shopping,Retail,d05bb230-5f44-4180-9f9b-323f72649552,508d94c4-3851-4b51-b0c0-4efcf6381fcb,Action Figure / Doll,Action Figure / Doll,Toys & Hobbies,Shopping,action figure / doll action figure / doll,"[-0.015342016, -0.06665123, 0.027730906, -0.014485563, -0.02620437, 0.018394252, 0.051076733, 0.0060139503, 0.029826336, 0.07496098, 0.08566069, -0.010734811, -0.0052721016, 0.06528328, 0.06540803..."
5,Travel,Channel - Travel,5d99d471-fb8a-496a-843b-f0a4a75a6e30,a63aa87d-7f06-47ca-bbc7-8197f773fc1e,Tour - Hokkaido & Tohoku - Thermal Bath,Tour - Hokkaido & Tohoku - Thermal Bath,Asia,Travel,tour hokkaido & tohoku thermal bath tour hokkaido & tohoku thermal bath,"[-0.011986596, 0.009610164, 0.015161306, 0.033717275, 0.0031568056, -0.05583473, 0.068553045, -0.012469813, -0.07547622, -0.012553563, -0.011668896, -0.07898682, -0.013184847, 0.098407656, 0.02999..."


In [63]:
def query_embedding_reduce(query_embedding):
    if query_embedding.shape[1] > 384:
        query_embedding_reduced = np.mean(query_embedding.reshape(-1, 2, 384), axis=1)
    else:
        query_embedding_reduced = query_embedding
    return query_embedding_reduced

df_[['name', 'guid', 'text', 'text_embeddings']].to_csv('models/category_embeding1.csv')
df_.head(1)

,attributes.v2_category_name,attributes.v1_category_name,parent,guid,description,name,header,category,text,text_embeddings
0,Shopping,Retail,0ed8f46e-2990-448c-9a8c-50665498a84c,7552494f-2a02-4bf8-91b3-ba34d90debdf,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Personal Care,Shopping,health & beauty deodorant & antiperspirant mens deodorant health & beauty deodorant & antiperspirant mens deodorant,"[-0.009740936, 0.01637392, 0.10180105, 0.015788225, 0.08613156, -0.05136446, 0.10834356, -0.023668014, -0.055397417, -0.020835223, 0.07027916, -0.053843062, -0.018749349, -0.05474668, 0.09677764, ..."


In [64]:
def get_top_similarity(query_embedding_reduced, combined_embeddings):
    similarities = cosine_similarity(query_embedding_reduced, combined_embeddings).flatten()
    closest_indices = np.argsort(similarities)[-10:]

    closest_rows = []
    for index in reversed(closest_indices):
        #if similarities[index] > 0.27:
        
        closest_rows.append([df_.iloc[index], similarities[index]])

    return closest_rows

### Test query -> category use Cosine similarity of category embedings and query embedings

In [65]:
def get_sim(query):
    start_time = time.time()
    query_embedding_reduced = query_embedding_reduce(model.encode(query).reshape(1, -1))
    print (f"Embeding time :{time.time() - start_time}")
    result = get_top_similarity(query_embedding_reduced, combined_embeddings)
    print (f"Total run time :{time.time() - start_time}")
    for row in result:
        print(f"Closest Category: <{row[0]['name']}> -> score {row[1]}")

In [66]:
get_sim(['massage', 'oil'])

Embeding time :0.16618013381958008
Total run time :0.17487692832946777
Closest Category: <Massage> -> score 0.7664586305618286
Closest Category: <Massage - Aroma Oil> -> score 0.6713477969169617
Closest Category: <Massage - Honey> -> score 0.6113001704216003
Closest Category: <Massage - Remedial> -> score 0.5985206961631775
Closest Category: <Massage - Relaxation> -> score 0.5876349210739136
Closest Category: <Massage Course> -> score 0.5853389501571655
Closest Category: <Massage - Hydro> -> score 0.5817310214042664
Closest Category: <Massage - Therapeutic> -> score 0.5809776782989502
Closest Category: <Massage - Sports> -> score 0.580046534538269
Closest Category: <Massage - Full Body> -> score 0.5704364776611328


In [67]:
get_sim(['oil'])

Embeding time :0.08987593650817871
Total run time :0.09698700904846191
Closest Category: <Oil Change> -> score 0.5857585668563843
Closest Category: <Condiment / Vinegar / Oil> -> score 0.45710885524749756
Closest Category: <Engine> -> score 0.4557563066482544
Closest Category: <Massage - Aroma Oil> -> score 0.4445527195930481
Closest Category: <Oil Change - Full Service> -> score 0.437126100063324
Closest Category: <Fries> -> score 0.4209892153739929
Closest Category: <Water> -> score 0.3868389129638672
Closest Category: <Health & Beauty - Shaving & Grooming - Shaving Oils - Womens> -> score 0.3792877793312073
Closest Category: <Fried Chicken> -> score 0.3775768280029297
Closest Category: <Consumables - Candles / Home Fragrance - Oil Diffuser> -> score 0.3759145736694336


In [68]:
get_sim(['change'])

Embeding time :0.05645012855529785
Total run time :0.0682680606842041
Closest Category: <eLearning - Change Management> -> score 0.44492727518081665
Closest Category: <Oil Change> -> score 0.42082688212394714
Closest Category: <Tire Change / Replacement> -> score 0.39209839701652527
Closest Category: <Tire / Tyre Change / Replacement> -> score 0.3848320245742798
Closest Category: <Moving> -> score 0.3161256015300751
Closest Category: <Knife> -> score 0.2904219627380371
Closest Category: <UK> -> score 0.2804293930530548
Closest Category: <Game> -> score 0.27997249364852905
Closest Category: <Behavior Modification> -> score 0.2758355140686035
Closest Category: <Basketball> -> score 0.26601549983024597


In [69]:
get_sim(['oil', 'change'])

Embeding time :0.05000591278076172
Total run time :0.05504202842712402
Closest Category: <Oil Change> -> score 0.623258113861084
Closest Category: <Oil Change - Full Service> -> score 0.43073558807373047
Closest Category: <Engine> -> score 0.4190903902053833
Closest Category: <Fries> -> score 0.4033411741256714
Closest Category: <Water> -> score 0.40010595321655273
Closest Category: <Gun> -> score 0.37405046820640564
Closest Category: <Cookie> -> score 0.3730608820915222
Closest Category: <Tank> -> score 0.37278133630752563
Closest Category: <Grill> -> score 0.3709569573402405
Closest Category: <Beer> -> score 0.3700261116027832


In [70]:
get_sim(['sauna', 'massage'])

Embeding time :0.2349398136138916
Total run time :0.27696681022644043
Closest Category: <Massage> -> score 0.8004816770553589
Closest Category: <Sauna> -> score 0.7759735584259033
Closest Category: <Massage - Relaxation> -> score 0.6885701417922974
Closest Category: <Massage - Full Body> -> score 0.6542151570320129
Closest Category: <Massage Course> -> score 0.6519696712493896
Closest Category: <Massage - Sports> -> score 0.6364989876747131
Closest Category: <Massage - Remedial> -> score 0.6352710127830505
Closest Category: <Massage - Custom> -> score 0.631118893623352
Closest Category: <Massage - Therapeutic> -> score 0.6181920766830444
Closest Category: <Massage - Californian> -> score 0.6137300729751587


In [71]:
get_sim(['oil', 'massage'])

Embeding time :0.03276419639587402
Total run time :0.03970599174499512
Closest Category: <Massage> -> score 0.7664586305618286
Closest Category: <Massage - Aroma Oil> -> score 0.6713477969169617
Closest Category: <Massage - Honey> -> score 0.6113001704216003
Closest Category: <Massage - Remedial> -> score 0.5985206961631775
Closest Category: <Massage - Relaxation> -> score 0.5876349210739136
Closest Category: <Massage Course> -> score 0.5853389501571655
Closest Category: <Massage - Hydro> -> score 0.5817310214042664
Closest Category: <Massage - Therapeutic> -> score 0.5809776782989502
Closest Category: <Massage - Sports> -> score 0.580046534538269
Closest Category: <Massage - Full Body> -> score 0.5704364776611328


In [72]:
get_sim(['massage', 'oil'])

Embeding time :0.056962013244628906
Total run time :0.06278300285339355
Closest Category: <Massage> -> score 0.7664586305618286
Closest Category: <Massage - Aroma Oil> -> score 0.6713477969169617
Closest Category: <Massage - Honey> -> score 0.6113001704216003
Closest Category: <Massage - Remedial> -> score 0.5985206961631775
Closest Category: <Massage - Relaxation> -> score 0.5876349210739136
Closest Category: <Massage Course> -> score 0.5853389501571655
Closest Category: <Massage - Hydro> -> score 0.5817310214042664
Closest Category: <Massage - Therapeutic> -> score 0.5809776782989502
Closest Category: <Massage - Sports> -> score 0.580046534538269
Closest Category: <Massage - Full Body> -> score 0.5704364776611328


In [73]:
get_sim(['valvoline', 'oil'])

Embeding time :0.03850293159484863
Total run time :0.052110910415649414
Closest Category: <Engine> -> score 0.4400181770324707
Closest Category: <Tank> -> score 0.41675952076911926
Closest Category: <Oil Change> -> score 0.40533214807510376
Closest Category: <Belgian> -> score 0.3923361897468567
Closest Category: <Massage - Aroma Oil> -> score 0.38942861557006836
Closest Category: <Grill> -> score 0.3778752088546753
Closest Category: <Pizza> -> score 0.37674736976623535
Closest Category: <Lamb> -> score 0.3750528395175934
Closest Category: <Condiment / Vinegar / Oil> -> score 0.36251482367515564
Closest Category: <Pie> -> score 0.36207276582717896


In [74]:
get_sim(['water'])

Embeding time :0.051867008209228516
Total run time :0.05759406089782715
Closest Category: <Water> -> score 0.9141244888305664
Closest Category: <Beverage> -> score 0.5128365159034729
Closest Category: <Juice> -> score 0.5127252340316772
Closest Category: <Drinks> -> score 0.5110993981361389
Closest Category: <Beer> -> score 0.5020281076431274
Closest Category: <Swimming> -> score 0.4985038638114929
Closest Category: <Swimming / Pool> -> score 0.47986841201782227
Closest Category: <Food, Beverages & Tobacco - Water> -> score 0.4740852415561676
Closest Category: <Water Delivery> -> score 0.4691886901855469
Closest Category: <Toys & Games - Water Sports> -> score 0.4364290237426758


In [75]:
get_sim(['water', 'parks'])

Embeding time :0.032466888427734375
Total run time :0.04172801971435547
Closest Category: <Water> -> score 0.6865593194961548
Closest Category: <Waterpark> -> score 0.6200923323631287
Closest Category: <Waterpark Resort> -> score 0.5546667575836182
Closest Category: <Swimming / Pool> -> score 0.5269086360931396
Closest Category: <Waterpark Resort - Beach> -> score 0.5177987813949585
Closest Category: <Hotel - Waterparks> -> score 0.5001858472824097
Closest Category: <Beer> -> score 0.49480360746383667
Closest Category: <Leisure Park> -> score 0.491523414850235
Closest Category: <Pool> -> score 0.4888867735862732
Closest Category: <Juice> -> score 0.48714131116867065


In [76]:
get_sim(['amc'])

Embeding time :0.05841684341430664
Total run time :0.06446194648742676
Closest Category: <Cinema / Movie Theater> -> score 0.47816646099090576
Closest Category: <Movies> -> score 0.41026613116264343
Closest Category: <Hotel - Hobart - Parks> -> score 0.4071919322013855
Closest Category: <Hotel - Hobart - Cruises> -> score 0.40591031312942505
Closest Category: <Motel> -> score 0.39845260977745056
Closest Category: <Hotel - Hobart - Casino Resorts> -> score 0.3862011134624481
Closest Category: <Cinema - Open Air> -> score 0.3853117823600769
Closest Category: <Tour - Hobart - Beach Vacations> -> score 0.3831312358379364
Closest Category: <Firewood> -> score 0.38291212916374207
Closest Category: <Circus Arts> -> score 0.38021788001060486


In [77]:
get_sim(['pilates'])

Embeding time :0.061918020248413086
Total run time :0.06709027290344238
Closest Category: <Pilates - Mat> -> score 0.7914905548095703
Closest Category: <Pilates - Equipment> -> score 0.724598228931427
Closest Category: <Sporting Goods - Yoga & Pilates> -> score 0.6646608114242554
Closest Category: <Sporting Goods - Yoga & Pilates - Reformers> -> score 0.6467440128326416
Closest Category: <Sporting Goods - Yoga & Pilates - Flexbands> -> score 0.6124844551086426
Closest Category: <Sporting Goods - Yoga & Pilates - Blocks> -> score 0.6045401096343994
Closest Category: <Sporting Goods - Yoga & Pilates - Chairs> -> score 0.5983500480651855
Closest Category: <Sporting Goods - Yoga & Pilates - Mats> -> score 0.5750695466995239
Closest Category: <Sporting Goods - Yoga & Pilates - Foam Wedges> -> score 0.5662339925765991
Closest Category: <Apparel & Accessories - Activewear - Yoga & Pilates> -> score 0.5535861253738403


In [78]:
get_sim(['ring'])

Embeding time :0.03652596473693848
Total run time :0.05116105079650879
Closest Category: <Ideeli - Accessories - Fine Metal Jewelry - Rings> -> score 0.5880773067474365
Closest Category: <Custom - Rings> -> score 0.5879087448120117
Closest Category: <Sporting Goods - Gymnastics Rings> -> score 0.564052164554596
Closest Category: <Apparel & Accessories - Rings> -> score 0.5504467487335205
Closest Category: <Ideeli - Accessories - Diamond Jewelry - Rings> -> score 0.5464150905609131
Closest Category: <Apparel & Accessories - Jewelry - Rings - Womens> -> score 0.5339648723602295
Closest Category: <Ideeli - Accessories - Diamond Jewelry - Engagement Rings> -> score 0.5337338447570801
Closest Category: <Apparel & Accessories - Jewelry - Rings - Mens> -> score 0.5306218266487122
Closest Category: <Ideeli - Accessories - Diamond Jewelry - Wedding Rings> -> score 0.5200458765029907
Closest Category: <Apparel & Accessories - Jewelry - Rings - Girls> -> score 0.5178504586219788


In [79]:
get_sim(['wheel'])

Embeding time :0.05566596984863281
Total run time :0.07596993446350098
Closest Category: <Wheels & Tires> -> score 0.6498978734016418
Closest Category: <Sporting Goods - Cycling - Wheels> -> score 0.619838535785675
Closest Category: <Wheel Restoration> -> score 0.5956562757492065
Closest Category: <Spinning> -> score 0.5694590210914612
Closest Category: <Sporting Goods - Skateboarding - Wheels> -> score 0.558051586151123
Closest Category: <Ferris Wheel / Panoramic Wheel> -> score 0.5545158982276917
Closest Category: <Bicycle> -> score 0.5220310688018799
Closest Category: <Automotive - Replacement Parts - Wheels> -> score 0.5215298533439636
Closest Category: <Wheel Alignment / Balancing> -> score 0.5174133777618408
Closest Category: <Scooter> -> score 0.5150357484817505


In [80]:
get_sim(['nail'])

Embeding time :0.03241896629333496
Total run time :0.0494999885559082
Closest Category: <Nail Care> -> score 0.6964738368988037
Closest Category: <Nail Design> -> score 0.6815297603607178
Closest Category: <Manicure> -> score 0.5911762118339539
Closest Category: <BB - Manicure> -> score 0.5683706998825073
Closest Category: <Home Improvement - Nails & Screws - Nails> -> score 0.528639018535614
Closest Category: <Health & Beauty - Nail Care> -> score 0.5273698568344116
Closest Category: <Home Improvement - Nails & Screws> -> score 0.4938981533050537
Closest Category: <Health & Beauty - Nail Care - Base Coat> -> score 0.4597338140010834
Closest Category: <Steak> -> score 0.4581037759780884
Closest Category: <Health & Beauty - Nail Care - Top Coat> -> score 0.4533841609954834


In [81]:
get_sim(['massage', 'palace'])

Embeding time :0.03133392333984375
Total run time :0.04588794708251953
Closest Category: <Massage> -> score 0.7586876749992371
Closest Category: <Massage - Sports> -> score 0.6406188607215881
Closest Category: <Massage Course> -> score 0.6380980610847473
Closest Category: <Massage - Relaxation> -> score 0.6254104375839233
Closest Category: <Massage - Oriental> -> score 0.602710485458374
Closest Category: <Massage - Full Body> -> score 0.6014814972877502
Closest Category: <Massage - Abhyanga> -> score 0.5884844064712524
Closest Category: <Massage - Honey> -> score 0.5851495265960693
Closest Category: <Massage - Custom> -> score 0.5834150314331055
Closest Category: <Massage - Therapeutic> -> score 0.5803464651107788


In [82]:
get_sim(['apple', 'cider'])

Embeding time :0.03548097610473633
Total run time :0.045125722885131836
Closest Category: <Apple Picking> -> score 0.549866795539856
Closest Category: <Fruit> -> score 0.5125828385353088
Closest Category: <Juice> -> score 0.4922872483730316
Closest Category: <Computer> -> score 0.4681187868118286
Closest Category: <Cutlery> -> score 0.45640164613723755
Closest Category: <Beer> -> score 0.43357759714126587
Closest Category: <Supermarket> -> score 0.4138566255569458
Closest Category: <Beverage> -> score 0.4034065008163452
Closest Category: <Electronics - Laptops - Business Mac> -> score 0.40302544832229614
Closest Category: <Beer / Wine> -> score 0.39736831188201904
